In [ ]:
# Import packages
import sys
import numpy as np
from pathlib import Path
src_path = str(Path.cwd().parent)
if src_path not in sys.path:
    sys.path.append(src_path)

import microscopy_analysis.d00_utils.utilities as utils
import microscopy_analysis.d00_utils.dirnames as dn
from microscopy_analysis.d01_init_proc import subtractbg as sb
from microscopy_analysis.d00_utils.dirnames import bg_sub_dirname, bg_sub_fig_dirname
from microscopy_analysis.d01_init_proc import applymask as am
from microscopy_analysis.d01_init_proc import vis_and_rescale as vr
from skimage import morphology

%matplotlib notebook
%matplotlib inline

import pandas as pd

from bioio import BioImage
import bioio_ome_tiff
import bioio_tifffile
from bioio.writers import OmeTiffWriter

from PIL import Image
from tqdm import tqdm

import matplotlib.pyplot as plt

In [ ]:
input_dirpath = Path(input())

In [ ]:
seg_cmpch = -2
seg_caaxch = -1

figs_dirpath = input_dirpath / dn.figs_dirname

for imgpath in tqdm(input_dirpath.glob('*.ome.tif')):

    img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
    img = img_file.data

    seg_cmp = (img[:, seg_cmpch, np.newaxis, :, :, :] > 0)
    seg_caax = (img[:, seg_caaxch, np.newaxis, :, :, :] > 0)
    fig = vr.create_cmp_fig(seg_caax, seg_cmp)
    fig = fig.astype('uint8')
    OmeTiffWriter.build_ome(data_shapes=[fig.data.shape], data_types=[fig.dtype], physical_pixel_sizes=[img_file.physical_pixel_sizes])
    OmeTiffWriter.save(fig, figs_dirpath / imgpath.name)

    